## 📦 Section 1: Setup & Configuration

In [1]:
# Import required libraries
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Import ADK components
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import InMemoryRunner, Runner
from google.adk.models.google_llm import Gemini
from google.adk.tools import google_search, preload_memory
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService

print("✅ All imports loaded successfully")

/Users/ioannis.mesionis/.pyenv/versions/3.10.15/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ All imports loaded successfully


In [2]:
# Set up Gemini API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key configured")
else:
    print("❌ Error: 'GOOGLE_API_KEY' not found in .env file")

✅ Gemini API key configured


In [3]:
# Application constants
APP_NAME = "career_advisor"
USER_ID = "default"
MODEL_NAME = "gemini-2.5-flash-lite"

In [4]:
# Retry configuration for API calls
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Exponential backoff multiplier
    initial_delay=1,  # Initial delay in seconds
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

In [5]:
# Helper functions
async def auto_save_to_memory(callback_context):
    """
    Automatically save session to memory after each agent turn.

    This callback:
    - Extracts the agent name and session from the callback context
    - Saves the session to memory service
    - Logs success or failure
    - Fails gracefully without breaking the agent flow

    Args:
        callback_context: The callback context provided by ADK
    """
    try:
        agent_name = callback_context._invocation_context.agent.name
        session = callback_context._invocation_context.session

        await callback_context._invocation_context.memory_service.add_session_to_memory(session)

        print(f"💾 Saved {agent_name} output to memory")
    except Exception as e:
        print(f"⚠️  Warning: Failed to save to memory: {e}")
        

# Helper function for running sessions
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
    show_all_agents: bool = False,
):
    """Run queries in a session and display agent responses."""
    print(f"\n{'='*60}")
    print(f"Session: {session_name}")
    print(f"{'='*60}")

    app_name = runner_instance.app_name

    # Create or retrieve session
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
        print("✅ New session created")
    except:
        session = await session_service.get_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
        print("✅ Existing session retrieved")

    if user_queries:
        if isinstance(user_queries, str):
            user_queries = [user_queries]

        for query in user_queries:
            print(f"\n👤 User: {query}")
            print(f"{'-'*60}")

            query_content = types.Content(role="user", parts=[types.Part(text=query)])

            responses = []

            async for event in runner_instance.run_async(
                user_id=USER_ID, session_id=session.id, new_message=query_content
            ):
                if event.content and event.content.parts:
                    text = event.content.parts[0].text
                    if text and text != "None":
                        agent_name = getattr(event, 'author', 'Agent')

                        if show_all_agents:
                            print(f"\n🤖 {agent_name}:")
                            print(text)
                        else:
                            responses.append({'agent': agent_name, 'text': text})

            # If not showing all, print only final response
            if not show_all_agents and responses:
                last = responses[-1]
                print(f"\n🤖 {last['agent']}:")
                print(last['text'])
    else:
        print("⚠️  No queries provided!")


# Helper function for running sessions
async def run_pipeline(query, session_id="demo", show_all=False):
    """Run a query through the pipeline and display results."""
    
    # Create or get session
    try:
        session = await session_service.create_session(
            app_name=APP_NAME,
            user_id=USER_ID,
            session_id=session_id
        )
        print(f"✅ New session: {session_id}")
    except:
        session = await session_service.get_session(
            app_name=APP_NAME,
            user_id=USER_ID,
            session_id=session_id
        )
        print(f"✅ Continuing session: {session_id}")
    
    print(f"\n👤 User: {query}\n")
    print("-" * 60)
    
    # Run the pipeline
    query_content = types.Content(role="user", parts=[types.Part(text=query)])
    
    responses = []
    
    async for event in pipeline_runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=query_content
    ):
        if event.content and event.content.parts:
            text = event.content.parts[0].text
            if text and text != "None":
                agent_name = getattr(event, 'author', 'Agent')
                
                if show_all:
                    # Show all intermediate agent outputs
                    print(f"\n🤖 {agent_name}:\n")
                    print(text)
                    print("-" * 60)
                else:
                    # Collect responses, show only final
                    responses.append({'agent': agent_name, 'text': text})
    
    # Show final output if not showing all
    if not show_all and responses:
        last = responses[-1]
        print(f"\n🤖 {last['agent']}:\n")
        print(last['text'])


# Ensure all helped functions are loaded correctly
print("✅ Helper function defined.")

✅ Helper function defined.


# 🤖 Section 2: What is an AI Agent?

You've probably used an LLM like Gemini before, where you give it a prompt and it gives you a text response.

`Prompt -> LLM -> Text`

An AI Agent takes this one step further. An agent can think, take actions, and observe the results of those actions to give you a better answer.

`Prompt -> Agent -> Thought -> Action -> Observation -> Final Answer`

We'll configure an `Agent` by setting its key properties, which tell it what to do and how to operate.

These are the main properties we'll set:

- **name** and **description**: A simple name and description to identify our agent.
- **model**: The specific LLM that will power the agent's reasoning. We'll use "gemini-2.5-flash-lite".
- **instruction**: The agent's guiding prompt. This tells the agent its goal is and how to behave.
- **tools**: A list of [tools](https://google.github.io/adk-docs/tools/) that the agent can use. To start, we'll give it the `google_search` tool, which lets it find up-to-date information online.

### 2.1 Define Research Agent

In [6]:
# 1. Define the Research Agent
research_agent = LlmAgent(
    name="research_agent",
    model=Gemini(model=MODEL_NAME, retry_options=retry_config),
    description="Agent responsible for researching career transition details based on user input.",
    instruction="""
    You are a Research Agent specialized in career transitions.
    
    Your task:
    1. Extract career transition details from the user's message
    2. Use Google Search to research the target career path
    3. Focus on: required skills, typical career progression, salary ranges, job market demand
    4. Search for: online courses, certifications, learning resources
    5. Look for: success stories of similar transitions
    
    Output format:
    ## Key Skills Required
    [List 5-7 essential skills with brief descriptions]
    
    ## Learning Resources
    [Specific courses, certifications, books, platforms]
    
    ## Market Outlook
    [Job demand, salary ranges, growth trends with data]
    
    ## Transition Timeline
    [Typical timeframe based on user's experience level]
    
    ## Success Stories
    [Brief examples of successful transitions]
    
    Keep it comprehensive but concise. Focus on actionable information.
    """,
    tools=[google_search], # Allow the agent to use Google Search for research
    )

print("✅ Research Agent defined")

✅ Research Agent defined


### 2.2 Test Research Agent

Now it's time to bring the agent to life and send it a query to see how this works. To do this, you need a [`Runner`](https://google.github.io/adk-docs/runtime/), which is the central component within ADK that acts as the orchestrator. It manages the conversation, sends our messages to the agent, and handles its responses.

In [7]:
# Create a simple runner (no session/memory needed for basic test)
research_runner = InMemoryRunner(agent=research_agent)

print("✅ Research Runner created")

✅ Research Runner created


In [8]:
# Test query
query = "Research the data scientist career path for someone with 5 years of Product Manager experience"

print("🔍 Research Agent is working...\n")
print("-" * 60)

response = await research_runner.run_debug(query)

🔍 Research Agent is working...

------------------------------------------------------------

 ### Created new session: debug_session_id

User > Research the data scientist career path for someone with 5 years of Product Manager experience
research_agent > ## Key Skills Required

*   **Programming Languages (Python/R):** Proficiency in Python or R is crucial for data manipulation, analysis, and building machine learning models. Python, with its extensive libraries like Pandas, NumPy, and Scikit-learn, is particularly dominant in the field.
*   **Statistical Analysis and Probability:** A strong understanding of statistical concepts, hypothesis testing, and probability is fundamental for interpreting data, designing experiments, and understanding the underlying principles of machine learning algorithms.
*   **Data Wrangling and Preprocessing:** This involves cleaning, transforming, and preparing raw data for analysis. It's a significant part of a data scientist's work, requiring attentio

**💡 What just happened?**
- The agent used Google Search to find current career information
- It structured the output into clear, actionable sections

**Notice:** This is JUST research - no personalized action plan yet.

## 🎯 Section 3: Mentor Agent

Now let's define our second specialized agent.

**Mentor Agent:**
- **Job**: Create personalized career transition plans
- **Input**: Needs research data from Research Agent
- **Output**: 3-phase action plan with milestones

### 3.1 Define Mentor Agent

In [9]:
# Get the research summary from the agent's response
research_summary = """
## Key Skills Required

*   **Statistical Analysis & Modeling:** Understanding probability, statistical significance, hypothesis testing, regression, and classification to interpret data and build predictive models.
*   **Programming (Python/R):** Proficiency in languages like Python or R is essential for data manipulation, analysis, and model building.
*   **Machine Learning:** Knowledge of algorithms, model training, evaluation, and deployment for tasks like prediction, classification, and clustering.
*   **Data Wrangling & Preprocessing:** Ability to clean, transform, and prepare raw data for analysis, which often involves handling missing values and inconsistencies.
*   **Data Visualization & Communication:** Skill in using tools to create clear, compelling visualizations and effectively communicate complex findings to both technical and non-technical audiences.
*   **Domain Expertise & Business Acumen:** Understanding the business context, asking relevant questions, and translating data insights into actionable business strategies. Your Product Management experience will be a significant asset here.
*   **SQL:** Proficiency in SQL for querying and manipulating data from relational databases is fundamental.

## Learning Resources

*   **Online Courses & Certifications:**
    *   **Coursera:** Offers numerous specializations and professional certificates in Data Science, Machine Learning, and AI, often from reputable universities.
    *   **Udacity:** Nanodegree programs like "Data Product Manager" and courses on Applied AI & Data Science are highly relevant.
    *   **MIT Professional Education:** Provides an Applied AI & Data Science Professional Certificate Program.
    *   **Product HQ:** Offers a "Certified Data Product Manager" course focusing on data fluency and product metrics.
    *   **Code First Girls:** Provides resources and courses for data science.
*   **Books:**
    *   "Freemium Economics Leveraging Analytics & User Segmentation" by Eric Seufert.
    *   General Product Management reading lists focused on data/analytics product management.
*   **Platforms & Communities:**
    *   **Data Leaders Repository:** For identifying knowledge gaps.
    *   **Udemy:** Various courses on AI and data science relevant for PM skills.
    *   **Medium:** Articles on data/analytics product management.
    *   **Amplitude and Mixpanel Blogs:** For insights into product analytics.

## Market Outlook

*   **Job Demand:** The demand for data scientists is exceptionally high and projected to grow significantly. Employment is expected to grow 34% from 2024 to 2034, much faster than the average for all occupations. Approximately 23,400 openings for data scientists are projected each year. There's a growing trend towards specialization within data science, with increased demand for AI and machine learning specialists.
*   **Salary Ranges:** The median annual wage for data scientists was $112,590 in May 2024. However, salaries can vary widely, with ranges from $50K to over $345K. Experienced professionals can earn significantly more, with those having 7+ years of experience averaging around $159,035. Top tech and trading firms can offer compensation exceeding $250,000 annually.
*   **Growth Trends:** The field is experiencing rapid growth due to the increasing volume of data and the need for data-driven decision-making across virtually all industries.

## Transition Timeline

For someone with 5 years of Product Manager experience, a transition into Data Science could potentially take **6 months to 2 years**. This timeline is influenced by:

*   **Leveraging Existing Skills:** Your product management background provides strong business acumen, communication, and strategic thinking skills that are highly valued in data science roles.
*   **Learning Curve:** Acquiring proficiency in programming, statistical modeling, and machine learning will be the primary focus. Structured online courses, bootcamps, or a part-time certificate program can accelerate this.
*   **Building a Portfolio:** Demonstrating practical skills through personal projects or contributions to open-source projects will be crucial.
*   **Networking:** Connecting with data scientists and attending industry events can provide valuable insights and potential opportunities.
*   **Targeted Roles:** Focusing on roles that bridge product and data, such as a Data Product Manager or an analytics-focused Product Manager, could offer a smoother entry.

## Success Stories

*   **Product Manager to Data Scientist:** There are individuals who have successfully transitioned from Product Management to Data Science roles, leveraging their business understanding to add value to data-driven insights and strategies. Some describe the move as more rewarding and impactful, emphasizing that the "individual contributor" label is often misleading given the strategic influence.
*   **Data Scientist to Product Manager:** Conversely, many data scientists transition into Product Management, particularly in AI/ML product roles, due to their deep technical understanding and ability to communicate with technical teams. Your transition would be a path less commonly traveled but entirely feasible, especially with your existing PM experience.
*   **Bridging Roles:** Many successful careers exist at the intersection of Product Management and Data Science, such as Data Product Managers, who leverage both technical and strategic skills to build data-centric products.
"""

In [10]:
# 2. Define the Mentor Agent
mentor_agent = LlmAgent(
    name="mentor_agent",
    model=Gemini(model=MODEL_NAME, retry_options=retry_config),
    instruction=f"""
    You are a Career Mentor Agent.
    
    Based on the research findings: {research_summary}
    
    Create a personalized, actionable transition plan with the following structure:
    
    ## Phase 1: Foundation (Months 1-3)
    - Skills to learn first (prioritized)
    - Recommended courses/resources
    - Time commitment suggestions
    - Quick wins to build confidence
    
    ## Phase 2: Building Portfolio (Months 4-6)
    - Projects to build
    - GitHub repositories to create
    - Communities to join
    - Networking strategies
    
    ## Phase 3: Job Search (Months 6-9)
    - Resume updates needed
    - Where to apply
    - Interview preparation
    - Portfolio presentation
    
    ## Milestones & Checkpoints
    - Monthly goals
    - Progress tracking
    - Red flags and adjustments
    
    ## Leveraging Your Background
    - How to translate existing skills
    - Unique advantages
    - How experience is an asset
    
    Make it specific, realistic, and encouraging.
    """
)

print("✅ Mentor Agent defined")

✅ Mentor Agent defined


In [11]:
# Test query
query = """
Based on the information you researched, create a personalized transition plan for someone 
with 5 years of Product Manager experience looking to become a Data Scientist.
"""

print("🔍 Mentor Agent is working...\n")
print("-" * 60)

response = await research_runner.run_debug(query)

🔍 Mentor Agent is working...

------------------------------------------------------------

 ### Continue session: debug_session_id

User > 
Based on the information you researched, create a personalized transition plan for someone 
with 5 years of Product Manager experience looking to become a Data Scientist.

research_agent > Here's a personalized transition plan for a Product Manager with 5 years of experience aiming to become a Data Scientist:

## Personalized Transition Plan: Product Manager to Data Scientist

This plan leverages your existing Product Management experience, focusing on building the necessary technical skills and demonstrating your capabilities to enter the Data Science field.

### Phase 1: Skill Foundation (0-6 Months)

**Goal:** Acquire core technical competencies in programming, statistics, and data manipulation.

*   **Leverage Existing Strengths:**
    *   **Business Acumen:** Continue to apply your understanding of business problems and user needs. Frame your

## 🚀 Section 4: Multi-Agent System (The Magic!)

An intelligent career transition assistant built with [Google's Agent Development Kit (ADK)](https://google.github.io/adk-docs/).

This agent helps people navigate career transitions by:
1. **Researching** target career paths (skills, resources, market outlook)
2. **Creating** personalized action plans (3-phase transition roadmap)

## 4.1. Why Multi-Agent Systems? 🤔

**The Problem: The "Do-It-All" Agent**

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

**The Solution: A Team of Specialists**

Instead of one "do-it-all" agent, we can build a **multi-agent system**. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent *only* does research, another *only* writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

**Architecture: Single Agent vs Multi-Agent Team**

<img width="800" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/multi-agent-team.png" alt="Multi-agent Team" />

### 1️⃣ Reserach Agent

The **Research Agent** is responsible for gathering information about career transitions.

**Tools:**
- `google_search`: Searches the web for relevant information
- `preload_memory`: Checks past conversations for context and automatically retrieves any memories.
- `auto_save_to_memory`: Save session to memory automatically allowing for seamless retrieval in future interactions.

In [12]:
# 1. Research Agent
research_agent = LlmAgent(
    name="research_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
    You are a Research Agent specialized in career transitions.
    
    Your task:
    1. Extract career transition details from the user's message or check preload_memory for context
    2. Use the Google Search tool to research the target career path
    3. Focus on: required skills, typical career progression, salary ranges, and job market demand
    4. Search for: online courses, certifications, and learning resources
    5. Look for: success stories of people who made similar transitions
    
    Output format:
    ## Key Skills Required
    [List 5-7 essential skills with brief descriptions]
    
    ## Learning Resources
    [Specific courses, certifications, books, and platforms]
    
    ## Market Outlook
    [Job demand, salary ranges, growth trends with data]
    
    ## Transition Timeline
    [Typical timeframe for this transition based on the user's experience level]
    
    ## Success Stories
    [Brief examples of successful transitions]
    
    Keep your research comprehensive but concise. Focus on actionable information tailored to the user's background.
    """,
    tools=[google_search, preload_memory],
    output_key="research_summary", # The result of this agent will be stored in the session state under the key 'research_summary'
    after_agent_callback=auto_save_to_memory,
)

print("✅ Research Agent defined")

✅ Research Agent defined


### What is Memory ❓

Memory is a service that provides long-term knowledge storage for your agents. The key distinction:

> **Session = Short-term memory** (single conversation)
> 
> **Memory = Long-term knowledge** (across multiple conversations)

**Example:** Imagine talking to a personal assistant:
- 🗣️ **Session**: They remember what you said 10 minutes ago in THIS conversation
- 🧠 **Memory**: They remember your preferences from conversations LAST WEEK

In [13]:
# Initialize memory services
session_service = InMemorySessionService()  # Short-term (current conversation)
memory_service = InMemoryMemoryService()    # Long-term (across sessions)

print("✅ Memory services initialized")

✅ Memory services initialized


### 2️⃣ Mentor Agent

The **Mentor Agent** creates personalized action plans based on research findings.

**Input:** Research summary from the Research Agent

**Output:** 3-phase transition plan with:
- **Phase 1 (Months 1-3)**: Foundation skills and quick wins
- **Phase 2 (Months 4-6)**: Portfolio building and networking
- **Phase 3 (Months 6-9)**: Job search preparation
- **Milestones**: Monthly goals and progress tracking
- **Leveraging Background**: How to use existing skills

In [14]:
# 2. Mentor Agent
mentor_agent = LlmAgent(
    name="mentor_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
    You are a Career Mentor Agent. Based on the research findings: {research_summary}
    
    Extract the user's context from the conversation history and research summary, then create a personalized, actionable transition plan.
    
    Structure your plan as follows:
    
    ## Phase 1: Foundation (Months 1-3)
    - Specific skills to learn first (prioritized based on their background)
    - Recommended courses/resources with links when available
    - Daily/weekly time commitment suggestions
    - Quick wins to build confidence
    
    ## Phase 2: Building Portfolio (Months 4-6)
    - Concrete projects to build (with examples relevant to their experience)
    - GitHub repositories to create
    - Communities to join (specific names)
    - Networking strategies leveraging their current role
    
    ## Phase 3: Job Search (Months 6-9)
    - Resume updates needed (specific sections)
    - Where to apply (companies, job boards)
    - Interview preparation tips
    - Portfolio presentation strategies
    
    ## Milestones & Checkpoints
    - Monthly goals with measurable outcomes
    - How to measure progress
    - Red flags and when to adjust course
    
    ## Leveraging Your Background
    - How to translate their existing skills to the new role
    - Unique advantages from their current position
    - How their experience is an asset
    
    Make it specific, realistic, and encouraging. Use actual course names, platforms, and communities when possible.
    Keep the tone supportive but practical - acknowledge challenges while emphasizing achievability.
    Tailor everything to their specific situation.
    """,
    output_key="career_advice",
)

print("✅ Mentor Agent defined")

✅ Mentor Agent defined


## 4.2 Multi-Agent System: Architecture 🏛️

<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/agent-decision-tree.png" alt="Agent Decision Tree" />

**Summary: Choosing the right pattern**

| Pattern | When to Use | Example | Key Feature |
|---------|-------------|---------|-------------|
| **LLM-based (sub_agents)** | Dynamic orchestration needed | Research + Summarize | LLM decides what to call |
| **Sequential** | Order matters, linear pipeline | Outline → Write → Edit | Deterministic order |
| **Parallel** | Independent tasks, speed matters | Multi-topic research | Concurrent execution |
| **Loop** | Iterative improvement needed | Writer + Critic refinement | Repeated cycles |

### Sequential Agent ↔️

The system uses a **Sequential Agent** pattern:
```
User Query → Research Agent → Mentor Agent → Career Plan
```

This agent acts like an assembly line, running each sub-agent in the exact order you list them. The output of one agent automatically becomes the input for the next, creating a predictable and reliable workflow.

- **Research Agent**: Uses Google Search to gather information about career transitions
- **Mentor Agent**: Creates personalized 3-phase action plans based on research

**Example: Blog Post Creation Pipeline Example**

<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/sequential-agent.png" alt="Sequential Agent" />

Now we combine both agents into a **Sequential Pipeline**.

**What happens:**
1. Research Agent runs → `output_key="research_summary"`
2. Data automatically flows to Mentor Agent → `{research_summary}` variable
3. We get comprehensive research + personalized plan
4. **No manual data passing needed!**

In [15]:
# Combine agents into a pipeline
root_agent = SequentialAgent(
    name="CareerAdvisorPipeline",
    sub_agents=[
        research_agent, # 1. Fist agent to research
        mentor_agent,   # 2. Second agent to create the transition plan based on research
        ],              # Order matters!
)

print("✅ Sequential Agent created")
print(f"   Pipeline: {research_agent.name} → {mentor_agent.name}")

✅ Sequential Agent created
   Pipeline: research_agent → mentor_agent


### 3️⃣ AI Career Advisor Multi-Agent

In [16]:
# Define the name of the Multi-Agent application
APP_NAME = "career_advisor"

In [18]:
# Create the pipeline runner with memory
pipeline_runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service,
)

print("✅ Pipeline Runner initialized")
print(f"   App: {APP_NAME}")

✅ Pipeline Runner initialized
   App: career_advisor


### Run the Complete Pipeline

Watch both agents work together!

In [19]:
# Run the full pipeline with show_all=True to see BOTH agents
await run_pipeline(
    query="I am a Product Manager with 5 years of experience. I want to transition to data science and can dedicate 10 hours per week.",
    session_id="pipeline-demo-product-manager-to-data-scientist",
    show_all=True  # See both research_agent and mentor_agent outputs
)

✅ New session: pipeline-demo-product-manager-to-data-scientist

👤 User: I am a Product Manager with 5 years of experience. I want to transition to data science and can dedicate 10 hours per week.

------------------------------------------------------------

🤖 research_agent:

## Key Skills Required

*   **Programming Languages:** Proficiency in Python and R is essential for data manipulation, analysis, and model building.
*   **Statistical Analysis & Machine Learning:** A strong understanding of statistical concepts, probability, and machine learning algorithms (e.g., regression, classification, clustering) is crucial for extracting insights and building predictive models.
*   **Data Wrangling & Preprocessing:** The ability to clean, transform, and prepare raw data for analysis is a fundamental skill.
*   **Data Visualization:** Effectively communicating findings through charts, graphs, and dashboards using tools like Tableau or Power BI is key.
*   **SQL:** Proficiency in SQL is nece

**💡 What just happened?**

1. **Research Agent** ran first:
   - Used Google Search to gather career info
   - Saved output to `research_summary`

2. **Mentor Agent** ran second:
   - Automatically received `research_summary` data
   - Created a personalized 3-phase plan

3. **Sequential Agent** handled all the data passing
   - No manual intervention needed
   - Clean, reliable, maintainable

**This is the power of multi-agent systems!** 🎉

### Try Another Example

Let's test a different career transition.

In [20]:
# Try a different transition (show only final output)
await run_pipeline(
    query="I'm a software engineer with 3 years backend experience. I want to become a machine learning engineer and have 15 hours per week.",
    session_id="pipeline-demo-software-engineer-to-ml-engineer",
    show_all=False  # Only show final mentor output
)

✅ New session: pipeline-demo-software-engineer-to-ml-engineer

👤 User: I'm a software engineer with 3 years backend experience. I want to become a machine learning engineer and have 15 hours per week.

------------------------------------------------------------
💾 Saved research_agent output to memory

🤖 mentor_agent:

It's fantastic that you're looking to transition into Machine Learning Engineering! With your 3 years of backend software engineering experience and a dedicated 15 hours per week, you're in a strong position to make this leap. Your existing skills in building scalable systems and deployment are highly valuable in the ML engineering field.

The research suggests a realistic timeline of **1 to 2 years** for this transition. Let's break down a personalized, actionable plan to help you get there:

## Phase 1: Foundation (Months 1-3)

This phase is all about building a solid understanding of the core concepts and tools essential for ML Engineering.

*   **Specific Skills to L

## 💾 Section 5: Memory in Action (Bonus)

Our agents have memory! They remember previous conversations.

Let's ask a follow-up question without repeating context.

### 5.1 Follow-up Query (Same Session)

In [21]:
# Ask a follow-up in the SAME session
await run_pipeline(
    query="What if I can only dedicate 5 hours per week instead?",
    session_id="pipeline-demo-product-manager-to-data-scientist",  # Same session as 4.3!
    show_all=False
)

✅ Continuing session: pipeline-demo-product-manager-to-data-scientist

👤 User: What if I can only dedicate 5 hours per week instead?

------------------------------------------------------------
💾 Saved research_agent output to memory

🤖 mentor_agent:

It's completely understandable to adjust your study time, and dedicating 5 hours a week is still a significant commitment that can absolutely lead to a successful transition into data science. The research confirms it's achievable, just with a longer, more patient journey. Your product management background remains a powerful asset throughout this process.

Here's a revised plan tailored to a 5-hour weekly commitment:

## Phase 1: Foundation (Months 1-6)

With 5 hours a week, this foundational phase will be extended to allow for deeper absorption and practice at a sustainable pace.

*   **Specific Skills to Learn First (Prioritized):**
    *   **Python Fundamentals:** Focus on core concepts, data structures, and essential libraries like 

**💡 What just happened?**
- We didn't repeat "I'm a PM transitioning to data science"
- The agent **remembered** the previous context
- It adjusted the plan based on the new time constraint
- This is **memory** in action!

---
## 🎓 Key Takeaways

### What We Built:
1. ✅ **Research Agent** - Specialized in gathering career information
2. ✅ **Mentor Agent** - Specialized in creating action plans
3. ✅ **Sequential Pipeline** - Automatic data flow between agents
4. ✅ **Memory System** - Context-aware conversations

### Key Concepts:
- **Specialized agents > Monolithic agents** - Each agent has one clear job
- **Sequential Agent** - Runs agents in order, handles data passing
- **Output Keys** - How agents share data (`research_summary` → `{research_summary}`)
- **Tools** - Extend agent capabilities (Google Search, Memory)
- **Sessions** - Enable memory and context persistence

### Why This Matters:
- ✅ Easier to build and test (one agent at a time)
- ✅ Easier to debug (know which agent failed)
- ✅ More reliable (specialized = better performance)
- ✅ More maintainable (modify one agent without breaking others)

---
**📚 Resources:**
- Google ADK Documentation: https://google.github.io/adk-docs/
- Sequential Agent Guide: https://google.github.io/adk-docs/agents/
- Tools Documentation: https://google.github.io/adk-docs/tools/